# Mamba-3 `siso-187m` smoke on free Colab (T4)

Runtime → Change runtime type → **T4 GPU**.

Colab often runs **Python 3.13**. This project pins `>=3.12,<3.13`, so we do **not** `pip install -e .`.
We put the clone on `PYTHONPATH` and use Colab's torch.

Backend is **torch** (pure PyTorch SISO). Official `mamba_ssm` kernels are skipped here.

Expect ~10–30 minutes the first time. **Disconnect and delete the runtime** when finished.

In [ ]:
import sys
import torch
print("python:", sys.version)
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
assert torch.cuda.is_available(), "Enable a T4 GPU: Runtime → Change runtime type → T4"

In [ ]:
# Clone the branch that has mamba3.train (boss_ranker).
import os
BRANCH = "boss_ranker"
REPO = "https://github.com/CornellDataScience/adhd.git"
if not os.path.isdir("/content/adhd"):
    !git clone --branch {BRANCH} --depth 1 {REPO} /content/adhd
else:
    !git -C /content/adhd fetch --depth 1 origin {BRANCH}
    !git -C /content/adhd checkout {BRANCH}
    !git -C /content/adhd pull --ff-only origin {BRANCH}
%cd /content/adhd
!git rev-parse --short HEAD
!ls mamba3/train | head

In [ ]:
# Colab's Python may be 3.13; skip installing the package (requires <3.13).
# Do NOT reinstall huggingface-hub/transformers — Colab already has them for Gradio/Diffusers.
# Only ensure einops; leave the preinstalled Hub stack alone.
import importlib.util
if importlib.util.find_spec("einops") is None:
    !pip -q install "einops>=0.8"

import os, sys
REPO_ROOT = "/content/adhd"
os.environ["PYTHONPATH"] = REPO_ROOT + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import transformers, huggingface_hub, einops
from mamba3.train import smoke  # noqa: F401
print("import ok")
print("transformers", transformers.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("einops", einops.__version__)

## Smoke: 5 steps on `siso-187m`

T4 has no bfloat16. Use fp16. Seq length 64 keeps free-tier memory safer for a first pass.

In [ ]:
import os
os.environ["PYTHONPATH"] = "/content/adhd"
!mkdir -p /content/smoke-187m
# FP32 weights + FP16 autocast (GradScaler cannot unscale FP16 parameters).
!git -C /content/adhd pull --ff-only origin boss_ranker
!PYTHONPATH=/content/adhd python -m mamba3.train.smoke \
  --model siso-187m \
  --backend torch \
  --device cuda \
  --dtype float32 \
  --steps 5 \
  --seq-len 64 \
  --batch-size 1 \
  --out /content/smoke-187m \
  --log /content/smoke-187m/smoke.jsonl

In [ ]:
import json
from pathlib import Path
log = Path("/content/smoke-187m/smoke.jsonl")
assert log.exists(), "smoke.jsonl missing — re-run the smoke cell"
rows = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
print("steps", len(rows))
print("first loss", rows[0]["loss"])
print("last loss", rows[-1]["loss"])
print("checkpoint", list(Path("/content/smoke-187m/final").iterdir()))
assert (Path("/content/smoke-187m/final") / "pytorch_model.bin").exists()
print("SMOKE OK")

## After Colab

1. **Runtime → Disconnect and delete runtime**
2. If this passed, next paid Runpod step uses `uv sync --extra cuda` and `--backend auto`
3. Do **not** start multi-day CPT on free Colab